# Hyperparameter Optimization (HPO) with Ray Tune, RAPIDS, and PyTorch on MNIST

This notebook demonstrates a practical HPO workflow for MNIST using:
- PyTorch for model training
- Ray Tune for distributed hyperparameter search
- RAPIDS (cuML, CuPy, cuDF) for GPU-accelerated data prep

The content is designed for Google Colab with GPU runtime enabled.

## 1. Install Dependencies

RAPIDS installation in Colab can vary with CUDA versions. The notebook includes a fallback path so the tutorial still runs if RAPIDS install fails.

In [ ]:
!pip -q install torch torchvision scikit-learn matplotlib pandas
!pip -q install "ray[tune]"

# RAPIDS for CUDA 12 (best effort for Colab).
!pip -q install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12 cupy-cuda12x || true

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split as sk_train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Runtime and RAPIDS Check

This verifies CUDA, initializes Ray, and checks RAPIDS imports.

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)

RAPIDS_AVAILABLE = True
try:
    import cudf
    import cuml
    import cupy as cp
    from cuml.model_selection import train_test_split as cu_train_test_split
    from cuml.preprocessing import StandardScaler as cuStandardScaler
    print('cuDF:', cudf.__version__)
    print('cuML:', cuml.__version__)
    print('CuPy:', cp.__version__)
except Exception as e:
    RAPIDS_AVAILABLE = False
    cp = None
    print('RAPIDS unavailable. CPU fallback is enabled.')
    print('Details:', e)

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, log_to_driver=False)
print('Ray initialized')

Torch: 2.1.0.post303
CUDA available: True
Device: cuda
cuDF: 23.12.01
cuML: 23.12.00
CuPy: 13.0.0


2026-07-23 12:12:57,560	INFO worker.py:1724 -- Started a local Ray instance.


Ray initialized


## 3. Load MNIST and Build Colab-Sized Splits

To keep HPO runtime practical, we train on a subset.

When RAPIDS is available, this notebook uses cuML train_test_split (GPU path). Otherwise it uses sklearn train_test_split (CPU path).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dir_path = '/content/drive/MyDrive/Accel_DS_RAPIDS/part4/data/'

print(f"Checking existence of directory: {dir_path}")
if os.path.exists(dir_path) and os.path.isdir(dir_path):
    print(f"The directory '{dir_path}' exists.")
else:
    print(f"Creating '{dir_path}'.")
    # Create data folder if it doesn't exist
    os.makedirs(dir_path, exist_ok=True)


In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST(root=dir_path, train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=dir_path, train=False, download=True, transform=transform)

X_train_full = train_set.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
y_train_full = train_set.targets.numpy().astype(np.int64)

X_test_full = test_set.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
y_test_full = test_set.targets.numpy().astype(np.int64)

if RAPIDS_AVAILABLE:
    print('Using cuML train_test_split on GPU...')
    rng = cp.random.RandomState(SEED)
    idx = rng.permutation(X_train_full.shape[0])[:25000]

    X_sub = cp.asarray(X_train_full)[idx]
    y_sub = cp.asarray(y_train_full)[idx]

    X_train_small, X_val_small, y_train_small, y_val_small = cu_train_test_split(
        X_sub, y_sub, test_size=0.2, random_state=SEED, shuffle=True
    )

    test_idx = rng.permutation(X_test_full.shape[0])[:5000]
    X_test_small = cp.asarray(X_test_full)[test_idx]
    y_test_small = cp.asarray(y_test_full)[test_idx]

    X_train_small = cp.asnumpy(X_train_small).astype(np.float32)
    X_val_small = cp.asnumpy(X_val_small).astype(np.float32)
    y_train_small = cp.asnumpy(y_train_small).astype(np.int64)
    y_val_small = cp.asnumpy(y_val_small).astype(np.int64)
    X_test_small = cp.asnumpy(X_test_small).astype(np.float32)
    y_test_small = cp.asnumpy(y_test_small).astype(np.int64)
else:
    print('Using sklearn train_test_split on CPU...')
    X_sub, _, y_sub, _ = sk_train_test_split(
        X_train_full, y_train_full,
        train_size=25000, stratify=y_train_full, random_state=SEED
    )

    X_train_small, X_val_small, y_train_small, y_val_small = sk_train_test_split(
        X_sub, y_sub,
        test_size=0.2, stratify=y_sub, random_state=SEED
    )

    X_test_small, _, y_test_small, _ = sk_train_test_split(
        X_test_full, y_test_full,
        train_size=5000, stratify=y_test_full, random_state=SEED
    )

print('Train:', X_train_small.shape, y_train_small.shape)
print('Val:  ', X_val_small.shape, y_val_small.shape)
print('Test: ', X_test_small.shape, y_test_small.shape)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:01<00:00, 7503859.48it/s] 


Extracting data/MNIST/raw/train-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 176081.35it/s]


Extracting data/MNIST/raw/train-labels-idx1-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:00<00:00, 1707653.65it/s]


Extracting data/MNIST/raw/t10k-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 2650692.75it/s]


Extracting data/MNIST/raw/t10k-labels-idx1-ubyte.gz to data/MNIST/raw

Using cuML train_test_split on GPU...
Train: (20000, 784) (20000,)
Val:   (5000, 784) (5000,)
Test:  (5000, 784) (5000,)


In [4]:
if RAPIDS_AVAILABLE:
    print('Scaling with cuML StandardScaler on GPU...')
    scaler = cuStandardScaler(with_mean=True, with_std=True)

    X_train_small = cp.asnumpy(scaler.fit_transform(cp.asarray(X_train_small))).astype(np.float32)
    X_val_small = cp.asnumpy(scaler.transform(cp.asarray(X_val_small))).astype(np.float32)
    X_test_small = cp.asnumpy(scaler.transform(cp.asarray(X_test_small))).astype(np.float32)
else:
    print('Scaling with sklearn StandardScaler on CPU...')
    scaler = StandardScaler()
    X_train_small = scaler.fit_transform(X_train_small).astype(np.float32)
    X_val_small = scaler.transform(X_val_small).astype(np.float32)
    X_test_small = scaler.transform(X_test_small).astype(np.float32)

print('Scaling complete')

Scaling with cuML StandardScaler on GPU...
Scaling complete


## 4. Define the PyTorch Model and Training Utilities

The model is a configurable MLP. Ray Tune will optimize hidden layer size, dropout, learning rate, batch size, and epochs.

In [5]:
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_1=256, hidden_2=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_2, 10)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
X_train_ref = X_train_small
y_train_ref = y_train_small
X_val_ref = X_val_small
y_val_ref = y_val_small

def train_mnist_tune(config):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    local_device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = MLP(
        hidden_1=config['hidden_1'],
        hidden_2=config['hidden_2'],
        dropout=config['dropout']
    ).to(local_device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()

    train_ds = TensorDataset(torch.from_numpy(X_train_ref), torch.from_numpy(y_train_ref))
    val_ds = TensorDataset(torch.from_numpy(X_val_ref), torch.from_numpy(y_val_ref))

    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

    for epoch in range(config['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(local_device)
            yb = yb.to(local_device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        total = 0
        val_loss_sum = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(local_device)
                yb = yb.to(local_device)
                logits = model(xb)

                val_loss = criterion(logits, yb)
                val_loss_sum += val_loss.item() * yb.size(0)

                preds = torch.argmax(logits, dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        tune.report(
            val_accuracy=correct / total,
            val_loss=val_loss_sum / total,
            epoch=epoch + 1
        )

## 5. Run Ray Tune Search

We use ASHA scheduling to stop poor configurations early and spend compute on better trials.

In [ ]:
search_space = {
    'hidden_1': tune.choice([128, 192, 256, 320, 384]),
    'hidden_2': tune.choice([64, 96, 128, 160, 192]),
    'dropout': tune.uniform(0.1, 0.5),
    'lr': tune.loguniform(1e-4, 5e-2),
    'batch_size': tune.choice([128, 256, 512]),
    'epochs': tune.choice([5, 8, 10, 12])
}

scheduler = ASHAScheduler(
    metric='val_accuracy',
    mode='max',
    max_t=12,
    grace_period=2,
    reduction_factor=2
)

resources = {'cpu': 2, 'gpu': 1 if torch.cuda.is_available() else 0}

run_name = f"ray_rapids_mnist_hpo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print('Run name:', run_name)

analysis = tune.run(
    train_mnist_tune,
    config=search_space,
    # metric='val_accuracy',
    # mode='max',
    num_samples=16,
    scheduler=scheduler,
    resources_per_trial=resources,
    name=run_name,
    verbose=1
)

best_config = analysis.get_best_config(metric='val_accuracy', mode='max')
print('Best config:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

ValueError: Tracked actor is not managed by this event manager: <TrackedActor 93464653640003026951265891640338193172>

## 6. Retrain Best Configuration and Evaluate

Now we retrain with best hyperparameters using train+validation data and evaluate on the test subset.

In [ ]:
X_train_final = np.concatenate([X_train_small, X_val_small], axis=0).astype(np.float32)
y_train_final = np.concatenate([y_train_small, y_val_small], axis=0).astype(np.int64)

final_model = MLP(
    hidden_1=best_config['hidden_1'],
    hidden_2=best_config['hidden_2'],
    dropout=best_config['dropout']
).to(device)

optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config['lr'])
criterion = nn.CrossEntropyLoss()

train_ds = TensorDataset(torch.from_numpy(X_train_final), torch.from_numpy(y_train_final))
test_ds = TensorDataset(torch.from_numpy(X_test_small), torch.from_numpy(y_test_small))

train_loader = DataLoader(train_ds, batch_size=best_config['batch_size'], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

for epoch in range(best_config['epochs']):
    final_model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = final_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

final_model.eval()
preds_all = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        logits = final_model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        preds_all.append(preds)

y_pred = np.concatenate(preds_all)
test_acc = accuracy_score(y_test_small, y_pred)
print(f'Test accuracy (best Ray Tune config): {test_acc:.4f}')
print('')
print(classification_report(y_test_small, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(y_test_small, y_pred, ax=ax, cmap='Blues', colorbar=False)
ax.set_title('MNIST Confusion Matrix (Ray Tune Best Model)')
plt.show()

## 7. Inspect Trial Results

Ray Tune stores full trial metadata. This table helps compare hyperparameters and outcomes.

In [ ]:
results_df = analysis.results_df
display_cols = [
    'val_accuracy', 'val_loss',
    'config.hidden_1', 'config.hidden_2', 'config.dropout',
    'config.lr', 'config.batch_size', 'config.epochs'
]
available_cols = [c for c in display_cols if c in results_df.columns]
results_df = results_df[available_cols].sort_values('val_accuracy', ascending=False)
results_df.head(10)

## 8. Summary

In this notebook you:
- Ran HPO on PyTorch MNIST with Ray Tune
- Used ASHA scheduling for early stopping of poor trials
- Used RAPIDS/cuML for GPU train_test_split and scaling when available
- Retrained and evaluated the best configuration on a held-out test set

Possible extensions:
- Increase num_samples for broader search
- Add Bayesian search (OptunaSearch or HyperOptSearch in Ray)
- Scale to multi-GPU Ray clusters